In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from diquark.data.loader import DataLoader
from diquark.features.feature_extractor import FeatureExtractor

from diquark.config.constants import PATH_DICT_ATLAS_136_75

In [ ]:
loader = DataLoader(PATH_DICT_ATLAS_136_75, index_stop=1000)

In [ ]:
datasets = loader.load_data()

In [ ]:
all_features = {}
for key, data in datasets.items():
    extractor = FeatureExtractor(n_jets=6)
    all_features[key] = extractor.compute_all(data)

In [ ]:
for k, v in all_features["SIG:Suu"].items():
    print(k, v.ndim, v.shape)
    print("")

In [ ]:
import yaml
from diquark.data.loader import DataLoader
from diquark.features.feature_extractor import FeatureExtractor
from diquark.data.preprocessor import Preprocessor
from diquark.models.gradient_boosting import GradientBoostingModel
from diquark.models.neural_network import NeuralNetworkModel
from diquark.config.constants import PATH_DICT_ATLAS_136_80

# Load configuration
with open("../diquark/config/default_settings.yaml", "r") as file:
    config = yaml.safe_load(file)

# Load data
loader = DataLoader(PATH_DICT_ATLAS_136_80, index_stop=config["data"]["index_stop"])
datasets = loader.load_data()

# Extract features
all_features = {}
for key, data in datasets.items():
    extractor = FeatureExtractor(n_jets=config["feature_extraction"]["n_jets"])
    all_features[key] = extractor.compute_all(data)


In [ ]:
with open("../diquark/config/default_settings.yaml", "r") as file:
    config = yaml.safe_load(file)

# Preprocess data
preprocessor = Preprocessor(config["preprocessing"])
X_train, X_test, y_train, y_test, df_train, df_test = preprocessor.prepare_data(all_features)

model = NeuralNetworkModel(config["models"]["neural_network"])
model.build(input_shape=X_train.shape[1])
training_results = model.train(X_train, y_train, X_test, y_test)

# Make predictions
y_pred = model.predict(X_test)

# Print some results
print("Training Results:", training_results)
# print("Feature Importances:", model.feature_importances())

# You can add more evaluation metrics here, such as accuracy, AUC, etc.
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc

print("AUC:", roc_auc_score(y_test, y_pred))
precision, recall, _ = precision_recall_curve(y_test, y_pred)
print("AUC-PR:", auc(recall, precision))

In [ ]:
model.model.summary()

In [ ]:
model = GradientBoostingModel(config["models"]["gradient_boosting"])
model.build(input_shape=X_train.shape[1])
training_results = model.train(X_train, y_train, X_test, y_test)

# Make predictions
y_pred = model.predict(X_test)

# Print some results
print("Training Results:", training_results)
print("Feature Importances:", model.feature_importances())

# You can add more evaluation metrics here, such as accuracy, AUC, etc.
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc

print("AUC:", roc_auc_score(y_test, y_pred))
precision, recall, _ = precision_recall_curve(y_test, y_pred)
print("AUC-PR:", auc(recall, precision))

# Jet Analysis

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go


# Function to extract values and errors from a string like "1.1681e-01 ± 1.0086e-02"
def extract_value_and_error(s):
    value, error = map(float, s.split("±"))
    return value, error

In [ ]:
df_6j = pd.read_csv("../jet_no_analysis/ATLAS_136_70_6j_5f/random_forest_counts_summary.csv")
df_7j = pd.read_csv("../jet_no_analysis/ATLAS_136_70_7j_5f/random_forest_counts_summary.csv")
df_8j = pd.read_csv("../jet_no_analysis/ATLAS_136_70_8j_5f/random_forest_counts_summary.csv")

In [ ]:
# Combine dataframes
dfs = [df_6j, df_7j, df_8j]
labels = ["6 jets", "7 jets", "8 jets"]

thresholds = [float(col) for col in df_6j.columns if col != "Process"][2:-1]
colors = ["rgb(31, 119, 180)", "rgb(255, 127, 14)", "rgb(44, 160, 44)"]

In [ ]:
# Create figure
fig = go.Figure()

for df, label, color in zip(dfs, labels, colors):
    # Extract S/B values and errors
    sb_row = df[df["Process"] == "S/B"].iloc[0]
    sb_values = []
    sb_errors = []
    for threshold in thresholds:
        value, error = extract_value_and_error(sb_row[str(threshold)])
        sb_values.append(value)
        sb_errors.append(error)

    # Convert to numpy arrays for easier calculations
    sb_values = np.array(sb_values)
    sb_errors = np.array(sb_errors)

    # Add trace for the central S/B values
    fig.add_trace(
        go.Scatter(
            x=thresholds, y=sb_values, mode="lines", name=f"S/B ({label})", line=dict(color=color)
        )
    )

    # Add error bands
    fig.add_trace(
        go.Scatter(
            x=thresholds + thresholds[::-1],
            y=np.concatenate([sb_values + sb_errors, (sb_values - sb_errors)[::-1]]),
            fill="toself",
            fillcolor=color.replace("rgb", "rgba").replace(")", ",0.2)"),  # Add transparency
            line=dict(color="rgba(255,255,255,0)"),
            hoverinfo="skip",
            name=f"Error band ({label})",
        )
    )

# Update layout
fig.update_layout(
    title="Signal to Background Ratio vs Threshold",
    xaxis_title="Threshold",
    yaxis_title="S/B Ratio",
    yaxis_type="log",
    legend_title="Legend",
    xaxis=dict(tickmode="array", tickvals=thresholds, ticktext=[f"{t:.3f}" for t in thresholds]),
    width=800,
    height=600,
)

# Show the plot
fig.show()

In [ ]:
df_6j